In [1]:
import os, cv2, torch
import numpy as np
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet101
from torch.utils.data import DataLoader
from pycocotools.coco import COCO
import matplotlib.pyplot as plt

In [2]:

class CocoSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_path, transforms=None):
        self.coco = COCO(ann_path)
        self.img_dir = img_dir
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = img_info['file_name']
        img = cv2.imread(os.path.join(self.img_dir, path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in anns:
            rle = self.coco.annToMask(ann)
            mask = np.maximum(mask, rle * ann['category_id'])
        if self.transforms:
            img = self.transforms(img)
            mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        return img, torch.tensor(mask, dtype=torch.long)

    def __len__(self):
        return len(self.ids)

In [13]:
from torch.utils.data import Subset
subset_indices = list(range(6000))
tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
train_dataset = CocoSegmentationDataset("mix-object-images", "mix-object-train.json", transforms=tfms)
train_dataset = Subset(train_dataset, subset_indices)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

loading annotations into memory...
Done (t=2.78s)
creating index...
index created!


In [8]:
max_label = 0
for i in range(10):  # sample first 10 images
    _, mask = train_dataset[i]
    max_label = max(max_label, mask.max().item())

print(f"Max label in dataset: {max_label}")


Max label in dataset: 2


In [9]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = deeplabv3_resnet101(pretrained=False, num_classes=3)
model.to(device)
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [14]:

num_epochs = 1
for epoch in range(num_epochs):
    running_loss = 0.0
    print(f"\nEpoch {epoch+1} starting...")
    for i, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        output = model(imgs)['out']
        loss = criterion(output, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 10 == 0:
            print(f"  Batch {i}/{len(train_loader)} - Loss: {loss.item():.4f}")
    print(f"Epoch {epoch+1} done! Avg Loss: {running_loss / len(train_loader):.4f}")

torch.save(model.state_dict(), "deeplabv3_mixobject.pth")


Epoch 1 starting...
  Batch 0/3000 - Loss: 0.2055
  Batch 10/3000 - Loss: 0.1487
  Batch 20/3000 - Loss: 0.1190
  Batch 30/3000 - Loss: 0.4551
  Batch 40/3000 - Loss: 0.1128
  Batch 50/3000 - Loss: 0.1037
  Batch 60/3000 - Loss: 0.0704
  Batch 70/3000 - Loss: 0.0684
  Batch 80/3000 - Loss: 0.3175
  Batch 90/3000 - Loss: 0.0801
  Batch 100/3000 - Loss: 0.1219
  Batch 110/3000 - Loss: 0.0622
  Batch 120/3000 - Loss: 0.0492
  Batch 130/3000 - Loss: 0.0505
  Batch 140/3000 - Loss: 0.0460
  Batch 150/3000 - Loss: 0.0409
  Batch 160/3000 - Loss: 0.0532
  Batch 170/3000 - Loss: 0.0559
  Batch 180/3000 - Loss: 0.7518
  Batch 190/3000 - Loss: 0.0655
  Batch 200/3000 - Loss: 0.0756
  Batch 210/3000 - Loss: 0.0464
  Batch 220/3000 - Loss: 0.0416
  Batch 230/3000 - Loss: 0.0453
  Batch 240/3000 - Loss: 0.0540
  Batch 250/3000 - Loss: 0.0418
  Batch 260/3000 - Loss: 0.0362
  Batch 270/3000 - Loss: 0.0356
  Batch 280/3000 - Loss: 0.0398
  Batch 290/3000 - Loss: 0.0532
  Batch 300/3000 - Loss: 0.050

In [15]:
torch.save(model.state_dict(), "deeplabv3_mixobject_epoch1.pth")


In [16]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from pycocotools.coco import COCO
import cv2
import numpy as np
class CocoSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_path, transforms=None):
        self.coco = COCO(ann_path)
        self.img_dir = img_dir
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = img_info['file_name']
        img = cv2.imread(os.path.join(self.img_dir, path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in anns:
            rle = self.coco.annToMask(ann)
            mask = np.maximum(mask, rle * ann['category_id'])
        if self.transforms:
            img = self.transforms(img)
            mask = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
        return img, torch.tensor(mask, dtype=torch.long)

    def __len__(self):
        return len(self.ids)

tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_dataset = CocoSegmentationDataset("mix-object-images", "mix-object-val.json", transforms=tfms)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

from torchvision.models.segmentation import deeplabv3_resnet101
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = deeplabv3_resnet101(pretrained=False, num_classes=3)  # 3 classes based on your dataset
model.load_state_dict(torch.load("deeplabv3_mixobject_epoch1.pth", map_location=device))
model.to(device)
model.eval()

def calculate_mean_iou(loader, model, num_classes=3):
    ious = []
    with torch.no_grad():
        for imgs, true_masks in loader:
            imgs = imgs.to(device)
            true_masks = true_masks.to(device)

            outputs = model(imgs)['out']
            preds = torch.argmax(outputs.squeeze(), dim=0)

            for cls in range(num_classes):
                pred_inds = preds == cls
                target_inds = true_masks.squeeze() == cls
                intersection = (pred_inds & target_inds).sum().float()
                union = (pred_inds | target_inds).sum().float()
                if union > 0:
                    ious.append((intersection / union).item())

    return sum(ious) / len(ious) if ious else 0.0

mean_iou = calculate_mean_iou(val_loader, model)
print(f"📊 Mean IoU on Validation Set: {mean_iou:.4f}")


loading annotations into memory...
Done (t=0.95s)
creating index...
index created!


/var/folders/c7/gjbfmqw50_n042sn1snttvw40000gn/T/ipykernel_38098/3310659407.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("deeplabv3_

📊 Mean IoU on Validation Set: 0.5313
